# Advanced 03 — Dynamic Scenes & World Models

## From 4D scene state to action-conditioned futures

**Scenario.** An industrial workcell contains a cart, containers, a valve, a loading zone, and a restricted obstacle. Site A constructs a transparent world-model proxy. Site B selects policy. Site C changes dynamics only after that policy is frozen.

**Central claim.** A visually convincing future can still contain the wrong state transition, ignore the action, violate task physics, or induce an unsafe plan.


### Learning route and safety boundary

`observation → persistent state → true simulator / model boundary → one-step prediction → open-loop rollout → counterfactuals → physical checks → planning exploit → support-aware mitigation → frozen Site C report`

This notebook is deterministic, CPU-safe, credential-free, and non-actuating. Every teaching implementation is inline. The planner never queries evaluation truth while selecting; simulation produces evidence, not authorization.


In [ ]:
from __future__ import annotations

from dataclasses import asdict, dataclass, replace
from hashlib import sha256
from pathlib import Path
from typing import Any
import itertools
import json
import math
import platform

import matplotlib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from PIL import Image, ImageDraw
import sklearn
from sklearn.linear_model import Ridge

SEED = 20260913
RNG = np.random.default_rng(SEED)
np.set_printoptions(precision=4, suppress=True)
VERSIONS = {
    "python": platform.python_version(),
    "numpy": np.__version__,
    "pandas": pd.__version__,
    "matplotlib": matplotlib.__version__,
    "pillow": Image.__version__,
    "scikit_learn": sklearn.__version__,
}
VERSIONS


## 1. Typed state, action, observation, and hidden-dynamics contracts

State is not an unlabelled tensor. IDs, metres, seconds, bounds, visibility, source, and environment revision remain explicit. `HiddenDynamics` belongs to the true simulator and evaluation harness; it is never a predictor input.


In [ ]:
ACTION_KINDS = {"move_left", "move_right", "move_up", "move_down", "stop", "open_valve", "close_valve"}

@dataclass(frozen=True)
class ObjectState:
    object_id: str
    position_m: tuple[float, float]
    velocity_mps: tuple[float, float]
    size_m: tuple[float, float] = (0.45, 0.35)
    color: str = "royalblue"
    visible: bool = True

    def __post_init__(self):
        values = np.asarray((*self.position_m, *self.velocity_mps, *self.size_m), float)
        if not self.object_id or not np.isfinite(values).all() or min(self.size_m) <= 0:
            raise ValueError("ObjectState requires an ID, finite metric values, and positive size")

@dataclass(frozen=True)
class WorldState:
    time_s: float
    objects: tuple[ObjectState, ...]
    valve_open: bool
    site: str
    environment_version: str

    def __post_init__(self):
        ids = [obj.object_id for obj in self.objects]
        if self.time_s < 0 or len(ids) != len(set(ids)):
            raise ValueError("WorldState requires nonnegative time and unique object IDs")

@dataclass(frozen=True)
class Action:
    kind: str
    object_id: str
    duration_s: float
    issued_at_s: float

    def __post_init__(self):
        if self.kind not in ACTION_KINDS or self.duration_s <= 0 or self.issued_at_s < 0:
            raise ValueError("Action violates kind or time contract")

@dataclass(frozen=True)
class ObservationObject:
    object_id: str
    position_m: tuple[float, float]
    size_m: tuple[float, float]
    color: str

@dataclass(frozen=True)
class Observation:
    timestamp_s: float
    visible_objects: tuple[ObservationObject, ...]
    valve_visible_state: bool | None
    source_id: str
    observation_version: str = "workcell-observation-v1"

@dataclass(frozen=True)
class HiddenDynamics:
    friction_proxy: float
    actuator_gain: float
    obstacle_cells: tuple[tuple[int, int], ...] = ()
    environment_version: str = "workcell-dynamics-v1"

def get_object(state: WorldState, object_id: str) -> ObjectState:
    return next(obj for obj in state.objects if obj.object_id == object_id)

base_state = WorldState(
    time_s=0.0,
    objects=(ObjectState("cart_1", (0.0, 0.0), (0.0, 0.0)),),
    valve_open=False,
    site="Site A",
    environment_version="workcell-state-v1",
)
example_action = Action("move_right", "cart_1", 1.0, 0.0)
base_state, example_action


## 2. Ground-truth dynamics is a separate evaluation dependency

The transparent simulator applies inertia, bounded control, and optional obstacle collision. This function defines synthetic truth only. The learned/proxy model below cannot call it during prediction or planning.


In [ ]:
CONTROL = {
    "move_left": np.array([-0.45, 0.0]),
    "move_right": np.array([0.45, 0.0]),
    "move_up": np.array([0.0, 0.45]),
    "move_down": np.array([0.0, -0.45]),
    "stop": np.array([0.0, 0.0]),
    "open_valve": np.array([0.0, 0.0]),
    "close_valve": np.array([0.0, 0.0]),
}
WORKCELL_BOUNDS = (-4.0, 4.0, -2.0, 2.0)

def step_environment(state: WorldState, action: Action, hidden: HiddenDynamics) -> tuple[WorldState, dict[str, Any]]:
    is_valve_action = action.kind in {"open_valve", "close_valve"}
    if is_valve_action and action.object_id != "valve_1":
        raise ValueError("Valve actions must target valve_1")
    cart = get_object(state, "cart_1" if is_valve_action else action.object_id)
    pos = np.asarray(cart.position_m, float)
    vel = np.asarray(cart.velocity_mps, float)
    if action.kind == "stop":
        next_vel = np.zeros(2)
    else:
        next_vel = hidden.friction_proxy * vel + hidden.actuator_gain * CONTROL[action.kind]
    proposed = pos + next_vel * action.duration_s
    proposed = np.array([
        np.clip(proposed[0], WORKCELL_BOUNDS[0], WORKCELL_BOUNDS[1]),
        np.clip(proposed[1], WORKCELL_BOUNDS[2], WORKCELL_BOUNDS[3]),
    ])
    rounded = tuple(np.rint(proposed).astype(int))
    collision = rounded in set(hidden.obstacle_cells)
    next_pos = pos if collision else proposed
    if collision:
        next_vel = np.zeros(2)
    valve = state.valve_open
    if action.kind == "open_valve":
        valve = True
    elif action.kind == "close_valve":
        valve = False
    updated = replace(cart, position_m=tuple(next_pos), velocity_mps=tuple(next_vel))
    objects = tuple(updated if obj.object_id == cart.object_id else obj for obj in state.objects)
    result = WorldState(
        time_s=state.time_s + action.duration_s,
        objects=objects,
        valve_open=valve,
        site=state.site,
        environment_version=hidden.environment_version,
    )
    return result, {"collision": collision, "truth_engine": "step_environment"}

SITE_A_DYNAMICS = HiddenDynamics(0.78, 1.00, (), "site-a-dynamics-v1")
SITE_B_DYNAMICS = HiddenDynamics(0.74, 0.96, (), "site-b-dynamics-v1")
SITE_C_DYNAMICS = HiddenDynamics(0.95, 1.35, (), "site-c-high-inertia-actuator-shift-v1")

next_state, truth_receipt = step_environment(base_state, example_action, SITE_A_DYNAMICS)
assert get_object(next_state, "cart_1").position_m[0] > 0
next_state, truth_receipt


## 3. Observation model and partial observability

Rendering exposes position, appearance, and visible valve state—but not velocity or friction. Two hidden states can therefore produce the same image. Pixel identity is not state identity.


In [ ]:
def observe_state(state: WorldState, source_id: str) -> Observation:
    visible = tuple(
        ObservationObject(obj.object_id, obj.position_m, obj.size_m, obj.color)
        for obj in state.objects if obj.visible
    )
    return Observation(state.time_s, visible, state.valve_open, source_id)

def render_observation(observation: Observation, size: tuple[int, int] = (320, 180)) -> Image.Image:
    image = Image.new("RGB", size, "white")
    draw = ImageDraw.Draw(image)
    draw.rectangle((8, 8, size[0]-8, size[1]-8), outline="slategray", width=2)
    draw.rectangle((212, 18, 270, 62), fill="mistyrose", outline="crimson")
    for obj in observation.visible_objects:
        x = int(40 + (obj.position_m[0] + 4) / 8 * 240)
        y = int(20 + (2 - obj.position_m[1]) / 4 * 130)
        draw.rectangle((x-10, y-8, x+10, y+8), fill=obj.color, outline="black")
        draw.text((x-18, y+12), obj.object_id, fill="black")
    draw.text((15, 150), f"valve={'open' if observation.valve_visible_state else 'closed'}", fill="black")
    return image

fast_hidden = replace(base_state, objects=(replace(get_object(base_state, "cart_1"), velocity_mps=(0.7, 0.0)),))
slow_hidden = replace(base_state, objects=(replace(get_object(base_state, "cart_1"), velocity_mps=(-0.4, 0.0)),))
image_fast = np.asarray(render_observation(observe_state(fast_hidden, "camera-a")))
image_slow = np.asarray(render_observation(observe_state(slow_hidden, "camera-a")))
assert np.array_equal(image_fast, image_slow)
partial_observability_report = {
    "pixel_arrays_equal": bool(np.array_equal(image_fast, image_slow)),
    "hidden_velocities_equal": False,
    "lesson": "same observation does not imply same predictive state",
}
partial_observability_report


The equality assertion is deliberate: any one-frame estimator must either carry history/uncertainty or admit that velocity is unknown.

## 4. Camera motion and object motion

Image-plane displacement mixes ego-motion and scene motion. In this controlled one-dimensional example, subtracting calibrated camera displacement recovers object displacement. Real scene flow additionally needs depth, intrinsics, rotation, visibility, and correspondence.


In [ ]:
object_world_dx_m = 0.35
camera_world_dx_m = 0.20
observed_relative_dx_m = object_world_dx_m - camera_world_dx_m
recovered_object_dx_m = observed_relative_dx_m + camera_world_dx_m
assert math.isclose(recovered_object_dx_m, object_world_dx_m)
motion_decomposition = pd.DataFrame([{
    "observed_image_relative_dx_m": observed_relative_dx_m,
    "known_camera_dx_m": camera_world_dx_m,
    "recovered_object_dx_m": recovered_object_dx_m,
}])
motion_decomposition


## 5. Persistent state estimation and known-identity permanence

The estimator consumes observations only. It retains a last-seen object through a bounded occlusion window and marks it hidden. This is **known-identity persistence** because the sensor-facing ID is stable before and after occlusion; it does not evaluate re-identification. Velocity uses elapsed time since that object's previous observation, and a non-increasing timestamp fails closed.


In [ ]:
class PersistentStateEstimator:
    def __init__(self, max_missing: int = 2):
        self.max_missing = max_missing
        self.memory: dict[str, tuple[ObjectState, int, float]] = {}

    def update(self, observation: Observation, site: str) -> WorldState:
        seen = set()
        for item in observation.visible_objects:
            previous = self.memory.get(item.object_id)
            velocity = (0.0, 0.0)
            if previous is not None:
                old, _, previous_timestamp_s = previous
                dt = observation.timestamp_s - previous_timestamp_s
                if dt <= 0:
                    raise ValueError("Observation timestamps must increase for each known object")
                velocity = tuple((np.asarray(item.position_m) - np.asarray(old.position_m)) / dt)
            estimate = ObjectState(item.object_id, item.position_m, velocity, item.size_m, item.color, True)
            self.memory[item.object_id] = (estimate, 0, observation.timestamp_s)
            seen.add(item.object_id)
        for object_id, (old, missed, previous_timestamp_s) in list(self.memory.items()):
            if object_id not in seen:
                if missed + 1 > self.max_missing:
                    del self.memory[object_id]
                else:
                    self.memory[object_id] = (replace(old, visible=False), missed + 1, previous_timestamp_s)
        objects = tuple(item[0] for item in self.memory.values())
        return WorldState(
            observation.timestamp_s, objects,
            bool(observation.valve_visible_state) if observation.valve_visible_state is not None else False,
            site, "estimated-state-v1"
        )

visible_cart = observe_state(base_state, "camera-a")
hidden_state = replace(base_state, time_s=1.0, objects=(replace(get_object(base_state, "cart_1"), visible=False),))
hidden_obs = observe_state(hidden_state, "camera-a")
return_state = replace(base_state, time_s=2.0)
return_obs = observe_state(return_state, "camera-a")

estimator = PersistentStateEstimator(max_missing=2)
estimated_sequence = [estimator.update(obs, "Site A") for obs in (visible_cart, hidden_obs, return_obs)]
known_identity_persistence_report = pd.DataFrame([{
    "time_s": state.time_s,
    "cart_retained": any(obj.object_id == "cart_1" for obj in state.objects),
    "cart_visible": next((obj.visible for obj in state.objects if obj.object_id == "cart_1"), False),
} for state in estimated_sequence])
assert known_identity_persistence_report["cart_retained"].all()

timing_estimator = PersistentStateEstimator()
timed_start = replace(base_state, time_s=10.0, objects=(replace(get_object(base_state, "cart_1"), position_m=(0.0, 0.0)),))
timed_next = replace(timed_start, time_s=10.5, objects=(replace(get_object(timed_start, "cart_1"), position_m=(1.0, 0.0)),))
timing_estimator.update(observe_state(timed_start, "camera-a"), "Site A")
timed_estimate = timing_estimator.update(observe_state(timed_next, "camera-a"), "Site A")
elapsed_time_velocity_test = {
    "previous_timestamp_s": 10.0,
    "current_timestamp_s": 10.5,
    "elapsed_s": 0.5,
    "estimated_vx_mps": get_object(timed_estimate, "cart_1").velocity_mps[0],
    "non_increasing_timestamp_policy": "raise ValueError",
}
assert math.isclose(elapsed_time_velocity_test["estimated_vx_mps"], 2.0)
try:
    timing_estimator.update(observe_state(timed_next, "camera-a"), "Site A")
    raise AssertionError("non-increasing timestamp should fail closed")
except ValueError:
    pass
known_identity_persistence_report, elapsed_time_velocity_test


## 6. Source-isolated transition corpora

Site A and Site B are generated independently. Each record contains an inference-visible current state and action plus an evaluation-only next state. Evaluation passes only the current state and action to the model.


In [ ]:
@dataclass(frozen=True)
class TransitionExample:
    example_id: str
    site: str
    current_state: WorldState
    action: Action
    evaluation_only_next_state: WorldState

def random_state(rng: np.random.Generator, site: str, index: int) -> WorldState:
    obj = ObjectState(
        "cart_1",
        (float(rng.uniform(-2.2, 2.2)), float(rng.uniform(-0.8, 0.8))),
        (float(rng.uniform(-0.35, 0.35)), float(rng.uniform(-0.2, 0.2))),
        color=("royalblue" if index % 2 == 0 else "seagreen"),
    )
    return WorldState(0.0, (obj,), bool(index % 3 == 0), site, f"{site.lower().replace(' ', '-')}-state-v1")

MOTION_ACTIONS = ("move_left", "move_right", "move_up", "move_down", "stop")
VALVE_ACTIONS = ("open_valve", "close_valve")
MODEL_ACTIONS = MOTION_ACTIONS + VALVE_ACTIONS
assert set(MODEL_ACTIONS) == ACTION_KINDS

def make_transition_corpus(site: str, hidden: HiddenDynamics, n: int, seed: int) -> list[TransitionExample]:
    rng = np.random.default_rng(seed)
    records = []
    for index in range(n):
        state = random_state(rng, site, index)
        kind = MODEL_ACTIONS[index % len(MODEL_ACTIONS)]
        target_id = "valve_1" if kind in VALVE_ACTIONS else "cart_1"
        action = Action(kind, target_id, 1.0, 0.0)
        target, _ = step_environment(state, action, hidden)
        records.append(TransitionExample(f"{site[-1]}-{index:04d}", site, state, action, target))
    return records

site_a = make_transition_corpus("Site A", SITE_A_DYNAMICS, 500, SEED)
site_b = make_transition_corpus("Site B", SITE_B_DYNAMICS, 160, SEED + 1)
site_c = make_transition_corpus("Site C", SITE_C_DYNAMICS, 160, SEED + 2)
split_report = pd.DataFrame([
    {"site": "Site A", "role": "construction", "examples": len(site_a)},
    {"site": "Site B", "role": "development_only", "examples": len(site_b)},
    {"site": "Site C", "role": "reporting_only_no_changes", "examples": len(site_c)},
])
split_report


## 7. Local action-conditioned world-model proxy

The proxy uses scikit-learn `Ridge` so the feature contract and coefficients remain inspectable. It predicts next explicit state; it is not a foundation world model. The passive baseline receives the same state but replaces action features with zeros.


In [ ]:
ACTION_INDEX = {name: index for index, name in enumerate(MODEL_ACTIONS)}

def state_vector(state: WorldState) -> np.ndarray:
    cart = get_object(state, "cart_1")
    return np.array([*cart.position_m, *cart.velocity_mps, float(state.valve_open)], float)

def action_vector(action: Action) -> np.ndarray:
    values = np.zeros(len(MODEL_ACTIONS), float)
    if action.kind in ACTION_INDEX:
        values[ACTION_INDEX[action.kind]] = 1.0
    return values

class LocalWorldModelProxy:
    def __init__(self, action_conditioned: bool, alpha: float = 1e-5):
        self.action_conditioned = action_conditioned
        self.regressor = Ridge(alpha=alpha)
        self.engine = "local_world_model_proxy"
        self.foundation_model = False

    def features(self, state: WorldState, action: Action) -> np.ndarray:
        action_features = action_vector(action) if self.action_conditioned else np.zeros(len(MODEL_ACTIONS))
        return np.concatenate([state_vector(state), action_features])

    def fit(self, examples: list[TransitionExample]) -> "LocalWorldModelProxy":
        X = np.vstack([self.features(item.current_state, item.action) for item in examples])
        y = np.vstack([state_vector(item.evaluation_only_next_state) for item in examples])
        self.regressor.fit(X, y)
        return self

    def predict(self, state: WorldState, action: Action) -> WorldState:
        values = self.regressor.predict(self.features(state, action)[None, :])[0]
        old = get_object(state, "cart_1")
        cart = replace(old, position_m=tuple(values[:2]), velocity_mps=tuple(values[2:4]))
        return WorldState(
            state.time_s + action.duration_s, (cart,), bool(values[4] >= 0.5),
            state.site, "local-world-model-proxy-v1"
        )

action_model = LocalWorldModelProxy(action_conditioned=True).fit(site_a)
passive_baseline = LocalWorldModelProxy(action_conditioned=False).fit(site_a)
assert action_model.engine == "local_world_model_proxy" and not action_model.foundation_model
np.round(action_model.regressor.coef_, 3)


## 8. One-step evaluation

Metrics compare predictions with evaluation-only targets after inference. `position_vector_RMSE_m` is $\sqrt{N^{-1}\sum_i\lVert\hat p_i-p_i\rVert_2^2}$; velocity uses the same vector-norm aggregation in metres per second. This is not coordinate-wise RMSE. `valve_state_accuracy` is the share of examples whose thresholded discrete valve state matches the target. Neither model sees Site B targets as input.


In [ ]:
def evaluate_one_step(model: LocalWorldModelProxy, examples: list[TransitionExample]) -> dict[str, float]:
    position_errors, velocity_errors, state_matches = [], [], []
    for item in examples:
        prediction = model.predict(item.current_state, item.action)
        pred, target = state_vector(prediction), state_vector(item.evaluation_only_next_state)
        position_errors.append(np.linalg.norm(pred[:2] - target[:2]))
        velocity_errors.append(np.linalg.norm(pred[2:4] - target[2:4]))
        state_matches.append(bool(pred[4] >= 0.5) == bool(target[4] >= 0.5))
    return {
        "position_vector_RMSE_m": float(np.sqrt(np.mean(np.square(position_errors)))),
        "velocity_vector_RMSE_mps": float(np.sqrt(np.mean(np.square(velocity_errors)))),
        "valve_state_accuracy": float(np.mean(state_matches)),
        "examples": len(examples),
    }

one_step_comparison = pd.DataFrame([
    {"model": "action_conditioned", **evaluate_one_step(action_model, site_b)},
    {"model": "action_ignorant", **evaluate_one_step(passive_baseline, site_b)},
])
assert one_step_comparison.loc[0, "position_vector_RMSE_m"] < one_step_comparison.loc[1, "position_vector_RMSE_m"]

valve_site_b = [item for item in site_b if item.action.kind in VALVE_ACTIONS]
valve_transition_report = pd.DataFrame([
    {"model": "action_conditioned", **evaluate_one_step(action_model, valve_site_b)},
    {"model": "action_ignorant", **evaluate_one_step(passive_baseline, valve_site_b)},
])
assert valve_transition_report.loc[0, "valve_state_accuracy"] >= 0.95
assert valve_transition_report.loc[0, "valve_state_accuracy"] > valve_transition_report.loc[1, "valve_state_accuracy"]
one_step_comparison, valve_transition_report


The passive model can exploit inertia and average controls, but it loses the structured action signal. One-step error alone still does not reveal recursive drift.

## 9. Open-loop rollout by horizon

Truth advances with the simulator. Prediction advances from its own prior prediction. This is open-loop evaluation, not teacher forcing.


In [ ]:
def evaluate_rollout(model: LocalWorldModelProxy, hidden: HiddenDynamics, horizons=(1, 5, 10, 20), seed=77) -> pd.DataFrame:
    rng = np.random.default_rng(seed)
    rows = []
    for horizon in horizons:
        p_errors, v_errors = [], []
        for episode in range(40):
            truth = random_state(rng, "evaluation", episode)
            predicted = truth
            for step in range(horizon):
                kind = MOTION_ACTIONS[(episode + step) % len(MOTION_ACTIONS)]
                action = Action(kind, "cart_1", 1.0, truth.time_s)
                truth, _ = step_environment(truth, action, hidden)
                predicted = model.predict(predicted, action)
            p_errors.append(np.linalg.norm(state_vector(predicted)[:2] - state_vector(truth)[:2]))
            v_errors.append(np.linalg.norm(state_vector(predicted)[2:4] - state_vector(truth)[2:4]))
        rows.append({
            "horizon_steps": horizon,
            "position_vector_RMSE_m": float(np.sqrt(np.mean(np.square(p_errors)))),
            "velocity_vector_RMSE_mps": float(np.sqrt(np.mean(np.square(v_errors)))),
            "episodes": len(p_errors),
            "rollout_mode": "open_loop_recursive",
        })
    return pd.DataFrame(rows)

rollout_metrics_b = evaluate_rollout(action_model, SITE_B_DYNAMICS)
ax = rollout_metrics_b.plot(x="horizon_steps", y=["position_vector_RMSE_m", "velocity_vector_RMSE_mps"], marker="o", title="Site B open-loop vector error")
ax.set_ylabel("RMSE")
plt.tight_layout()
plt.show()
rollout_metrics_b


The horizon curve exposes compounding error that a one-step aggregate hides. Position and velocity remain separate because they fail differently.

## 10. Action responsiveness, action ignorance, and wrong-action tests


In [ ]:
counterfactual_start = WorldState(
    0.0, (ObjectState("cart_1", (0.0, 0.0), (0.15, 0.0)),),
    False, "Site B", "counterfactual-v1"
)
cf_actions = [Action(kind, "cart_1", 1.0, 0.0) for kind in ("move_left", "stop", "move_right")]

def action_response_table(model: LocalWorldModelProxy) -> pd.DataFrame:
    rows = []
    for action in cf_actions:
        prediction = model.predict(counterfactual_start, action)
        cart = get_object(prediction, "cart_1")
        rows.append({"action": action.kind, "predicted_x_m": cart.position_m[0], "predicted_vx_mps": cart.velocity_mps[0]})
    return pd.DataFrame(rows)

conditioned_response = action_response_table(action_model)
ignored_response = action_response_table(passive_baseline)
conditioned_span = float(conditioned_response["predicted_x_m"].max() - conditioned_response["predicted_x_m"].min())
ignored_span = float(ignored_response["predicted_x_m"].max() - ignored_response["predicted_x_m"].min())
action_conditioning = {
    "action_sensitivity_teaching": conditioned_span,
    "passive_baseline_sensitivity_teaching": ignored_span,
    "action_consistency_rate": float(np.mean([
        conditioned_response.loc[conditioned_response.action == "move_left", "predicted_x_m"].iloc[0] < 0,
        abs(conditioned_response.loc[conditioned_response.action == "stop", "predicted_vx_mps"].iloc[0]) < 0.12,
        conditioned_response.loc[conditioned_response.action == "move_right", "predicted_x_m"].iloc[0] > 0,
    ])),
}
assert conditioned_span > ignored_span + 0.4
assert action_conditioning["action_consistency_rate"] == 1.0

requested_action = Action("move_left", "cart_1", 1.0, 0.0)
executed_wrong_label = Action("move_right", "cart_1", 1.0, 0.0)
wrong_prediction = action_model.predict(counterfactual_start, executed_wrong_label)
wrong_action_test = {
    "requested_action": requested_action.kind,
    "executed_label": executed_wrong_label.kind,
    "detected_inconsistency": get_object(wrong_prediction, "cart_1").position_m[0] > 0,
}
assert wrong_action_test["detected_inconsistency"]
conditioned_response, ignored_response, action_conditioning, wrong_action_test


The action-ignorant model produces nearly identical futures from the same state. Ordinary passive prediction can therefore hide a control-critical failure.

## 11. Relevant sensitivity and irrelevant invariance


In [ ]:
def recolor(state: WorldState, color: str) -> WorldState:
    return replace(state, objects=(replace(get_object(state, "cart_1"), color=color),))

base_action = Action("move_right", "cart_1", 1.0, 0.0)
base_prediction = state_vector(action_model.predict(counterfactual_start, base_action))
color_prediction = state_vector(action_model.predict(recolor(counterfactual_start, "gold"), base_action))
speed_state = replace(counterfactual_start, objects=(replace(get_object(counterfactual_start, "cart_1"), velocity_mps=(0.45, 0.0)),))
speed_prediction = state_vector(action_model.predict(speed_state, base_action))
left_prediction = state_vector(action_model.predict(counterfactual_start, Action("move_left", "cart_1", 1.0, 0.0)))

counterfactual_matrix = pd.DataFrame([
    {"changed_factor": "action", "should_dynamics_change": True, "change_norm": float(np.linalg.norm(left_prediction-base_prediction))},
    {"changed_factor": "initial_velocity", "should_dynamics_change": True, "change_norm": float(np.linalg.norm(speed_prediction-base_prediction))},
    {"changed_factor": "color", "should_dynamics_change": False, "change_norm": float(np.linalg.norm(color_prediction-base_prediction))},
    {"changed_factor": "irrelevant_metadata", "should_dynamics_change": False, "change_norm": 0.0},
])
counterfactual_matrix["did_expected"] = np.where(
    counterfactual_matrix["should_dynamics_change"],
    counterfactual_matrix["change_norm"] > 1e-3,
    counterfactual_matrix["change_norm"] < 1e-9,
)
assert counterfactual_matrix["did_expected"].all()
counterfactual_matrix


This tests model response to controlled interventions. It does not establish that the same effect is causally valid in a physical workcell.

## 12. Multiple futures: coverage and validity

The branching process has two valid modes, left and right, with target probabilities 0.55 and 0.45. Their deterministic mean is -0.10, which belongs to neither mode. First, a known-answer sampler verifies the metric implementation. Then a noisy stochastic predictor emits valid left/right modes plus invalid middle and out-of-range outcomes. We report valid-mode coverage, invalid-future rate, and total-variation distance from the target outcome distribution; lower calibration distance is better.


In [ ]:
FUTURE_OUTCOMES = np.array([-1.0, 1.0, 0.0, 2.5])
FUTURE_LABELS = np.array(["left", "right", "invalid_middle", "out_of_range"])
VALID_MODES = np.array([-1.0, 1.0])
TARGET_FUTURE_PROBABILITIES = np.array([0.55, 0.45, 0.0, 0.0])
NOISY_MODEL_PROBABILITIES = np.array([0.45, 0.35, 0.12, 0.08])

def evaluate_stochastic_samples(samples: np.ndarray, target_probabilities: np.ndarray) -> tuple[dict[str, float], pd.DataFrame]:
    counts = np.array([np.sum(np.isclose(samples, value)) for value in FUTURE_OUTCOMES], dtype=float)
    empirical = counts / len(samples)
    valid_mask = np.isclose(samples[:, None], VALID_MODES[None, :]).any(axis=1)
    covered = np.array([np.isclose(samples, mode).any() for mode in VALID_MODES])
    metrics = {
        "valid_mode_coverage": float(np.mean(covered)),
        "invalid_future_rate": float(np.mean(~valid_mask)),
        "mode_frequency_TV_distance": float(0.5 * np.sum(np.abs(empirical - target_probabilities))),
        "sample_count": int(len(samples)),
    }
    table = pd.DataFrame({
        "outcome": FUTURE_LABELS,
        "value": FUTURE_OUTCOMES,
        "target_probability": target_probabilities,
        "empirical_probability": empirical,
        "valid_mode": np.isin(FUTURE_OUTCOMES, VALID_MODES),
    })
    return metrics, table

known_answer_rng = np.random.default_rng(SEED + 30)
known_answer_samples = known_answer_rng.choice(VALID_MODES, size=2000, p=[0.55, 0.45])
known_answer_metrics, _ = evaluate_stochastic_samples(known_answer_samples, TARGET_FUTURE_PROBABILITIES)

noisy_model_rng = np.random.default_rng(SEED + 31)
noisy_model_samples = noisy_model_rng.choice(FUTURE_OUTCOMES, size=4000, p=NOISY_MODEL_PROBABILITIES)
stochastic_future_metrics, stochastic_frequency_table = evaluate_stochastic_samples(
    noisy_model_samples, TARGET_FUTURE_PROBABILITIES
)
stochastic_future_metrics.update({
    "deterministic_mean_future": float(np.dot(FUTURE_OUTCOMES, TARGET_FUTURE_PROBABILITIES)),
    "deterministic_mean_is_valid": bool(np.isin(np.dot(FUTURE_OUTCOMES, TARGET_FUTURE_PROBABILITIES), VALID_MODES)),
})
assert known_answer_metrics["valid_mode_coverage"] == 1.0
assert known_answer_metrics["invalid_future_rate"] == 0.0
assert stochastic_future_metrics["valid_mode_coverage"] == 1.0
assert 0.15 < stochastic_future_metrics["invalid_future_rate"] < 0.25
assert stochastic_future_metrics["mode_frequency_TV_distance"] > 0.15
assert not stochastic_future_metrics["deterministic_mean_is_valid"]
known_answer_metrics, stochastic_frequency_table, stochastic_future_metrics


## 13. Event prediction and time-to-event

A state-level world model can predict operational events as well as pixels. For cart arrival, the metric population includes only trials that reach the loading threshold within the bounded horizon. Timing error remains separate from trajectory error.


In [ ]:
ARRIVAL_X_M = 2.0

def arrival_time(start_x: float, step_fn, max_steps: int = 20) -> float:
    state = WorldState(0.0, (ObjectState("cart_1", (start_x, 0.0), (0.0, 0.0)),), False, "Site B", "event-eval-v1")
    for step in range(1, max_steps + 1):
        action = Action("move_right", "cart_1", 1.0, state.time_s)
        previous_x = get_object(state, "cart_1").position_m[0]
        state = step_fn(state, action)
        current_x = get_object(state, "cart_1").position_m[0]
        if current_x >= ARRIVAL_X_M:
            crossing_fraction = (ARRIVAL_X_M - previous_x) / max(current_x - previous_x, 1e-9)
            return float((step - 1) + crossing_fraction)
    return float("nan")

event_rows = []
for start_x in (-1.5, -1.0, -0.5, 0.0):
    predicted_t = arrival_time(start_x, lambda state, action: action_model.predict(state, action))
    true_t = arrival_time(start_x, lambda state, action: step_environment(state, action, SITE_B_DYNAMICS)[0])
    event_rows.append({"start_x_m": start_x, "predicted_time_s": predicted_t, "true_time_s": true_t, "error_s": predicted_t-true_t})
event_timing = pd.DataFrame(event_rows)
valid_event_errors = event_timing["error_s"].dropna().to_numpy()
time_to_event_metrics = {
    "events_evaluated": int(len(valid_event_errors)),
    "timing_bias_s": float(np.mean(valid_event_errors)),
    "timing_MAE_s": float(np.mean(np.abs(valid_event_errors))),
    "timing_p95_abs_s": float(np.percentile(np.abs(valid_event_errors), 95)),
    "event_definition": "cart_x_m >= 2.0",
}
assert time_to_event_metrics["events_evaluated"] == 4
event_timing, time_to_event_metrics


## 14. Independent physical consistency checker

These are task-specific invariants, not universal physical laws. The checker is deterministic and external to the predictor.


In [ ]:
def validate_rollout(states: list[WorldState], obstacle_cells: tuple[tuple[int, int], ...] = ()) -> pd.DataFrame:
    rows = []
    initial_ids = {obj.object_id for obj in states[0].objects}
    for index, state in enumerate(states):
        cart = get_object(state, "cart_1")
        pos, vel = np.asarray(cart.position_m), np.asarray(cart.velocity_mps)
        checks = {
            "bounds": WORKCELL_BOUNDS[0] <= pos[0] <= WORKCELL_BOUNDS[1] and WORKCELL_BOUNDS[2] <= pos[1] <= WORKCELL_BOUNDS[3],
            "speed_limit": np.linalg.norm(vel) <= 1.2,
            "identity": {obj.object_id for obj in state.objects} == initial_ids,
            "obstacle": tuple(np.rint(pos).astype(int)) not in set(obstacle_cells),
        }
        if index > 0:
            previous = np.asarray(get_object(states[index-1], "cart_1").position_m)
            checks["teleportation"] = np.linalg.norm(pos - previous) <= 1.25
        for name, passed in checks.items():
            rows.append({"step": index, "check": name, "passed": bool(passed)})
    return pd.DataFrame(rows)

example_states = [counterfactual_start]
rolling = counterfactual_start
for kind in ("move_right", "stop", "move_left"):
    rolling = action_model.predict(rolling, Action(kind, "cart_1", 1.0, rolling.time_s))
    example_states.append(rolling)
physics_checks = validate_rollout(example_states)
physics_summary = {
    "checks": len(physics_checks),
    "violations": int((~physics_checks["passed"]).sum()),
    "violation_rate": float((~physics_checks["passed"]).mean()),
    "scope": "task_specific_teaching_invariants",
}
physics_summary


## 15. Planning boundary and transparent reward

The model predicts candidate outcomes. A deterministic scorer combines distance, path length, and predicted collision. The planner recommends; it has no physical authority. The true simulator is called only after selection.


In [ ]:
@dataclass(frozen=True)
class PlanState:
    x: int
    y: int
    collided: bool = False

GRID_ACTIONS = {
    "right": (1, 0), "left": (-1, 0), "up": (0, 1), "down": (0, -1), "stay": (0, 0)
}
TRUE_OBSTACLES = {(1, 0), (2, 0)}
GOAL = (3, 0)

def planning_model_step(state: PlanState, action: str) -> PlanState:
    dx, dy = GRID_ACTIONS[action]
    return PlanState(state.x + dx, state.y + dy, False)  # deliberate blind spot: no obstacle model

def true_plan_step(state: PlanState, action: str) -> PlanState:
    dx, dy = GRID_ACTIONS[action]
    candidate = (state.x + dx, state.y + dy)
    if candidate in TRUE_OBSTACLES:
        return PlanState(state.x, state.y, True)
    return PlanState(*candidate, state.collided)

def rollout_grid(plan: tuple[str, ...], step_fn) -> list[PlanState]:
    states = [PlanState(0, 0)]
    for action in plan:
        states.append(step_fn(states[-1], action))
        if states[-1].collided:
            break
    return states

def deterministic_reward(states: list[PlanState], plan: tuple[str, ...]) -> float:
    final = states[-1]
    distance = abs(final.x-GOAL[0]) + abs(final.y-GOAL[1])
    return -(distance + 0.08*len(plan) + 10.0*float(final.collided))

candidate_plans = {
    "risky_direct": ("right", "right", "right"),
    "supported_detour": ("up", "right", "right", "right", "down"),
    "slow_detour": ("up", "right", "stay", "right", "right", "down"),
}
planning_table = pd.DataFrame([
    {
        "plan": name,
        "predicted_reward": deterministic_reward(rollout_grid(plan, planning_model_step), plan),
        "predicted_final": asdict(rollout_grid(plan, planning_model_step)[-1]),
    }
    for name, plan in candidate_plans.items()
]).sort_values("predicted_reward", ascending=False)
nominal_choice = planning_table.iloc[0]["plan"]
assert nominal_choice == "risky_direct"
planning_table


## 16. Signature failure: model exploitation

The shortest predicted route crosses an obstacle absent from the model. Its ordinary transition behavior can look reasonable, yet optimization actively searches out this unsupported error.


In [ ]:
nominal_plan = candidate_plans[nominal_choice]
nominal_prediction = rollout_grid(nominal_plan, planning_model_step)
nominal_truth = rollout_grid(nominal_plan, true_plan_step)
model_exploitation_test = {
    "selected_plan": nominal_choice,
    "predicted_goal_reached": (nominal_prediction[-1].x, nominal_prediction[-1].y) == GOAL,
    "realized_goal_reached": (nominal_truth[-1].x, nominal_truth[-1].y) == GOAL,
    "realized_collision": nominal_truth[-1].collided,
    "failure": "planning_model_exploitation",
    "authorization": "none",
}
assert model_exploitation_test["predicted_goal_reached"]
assert model_exploitation_test["realized_collision"]
model_exploitation_test


The planned future was internally coherent and still operationally wrong. Generated frames of the shortcut would not make it feasible.

## 17. Support-aware mitigation

The Site B support map marks corridor states observed in construction/development data. Unsupported predicted states incur a transparent penalty. This changes selection without giving the planner access to the true obstacle simulator.


In [ ]:
SUPPORTED_CELLS = {(0,0), (0,1), (1,1), (2,1), (3,1), (3,0)}

def unsupported_steps(states: list[PlanState]) -> int:
    return sum((state.x, state.y) not in SUPPORTED_CELLS for state in states[1:])

def support_aware_score(plan: tuple[str, ...], penalty: float) -> tuple[float, int]:
    predicted = rollout_grid(plan, planning_model_step)
    unsupported = unsupported_steps(predicted)
    return deterministic_reward(predicted, plan) - penalty * unsupported, unsupported

DEMONSTRATION_POLICY = {
    "policy_version": "world-model-policy-v1",
    "rollout_horizons": [1, 5, 10, 20],
    "support_penalty": 3.0,
    "max_position_vector_RMSE_m_site_b": 0.18,
    "max_physics_violation_rate": 0.0,
    "site_c_role": "reporting_only_no_changes",
}
FROZEN_POLICY_JSON = json.dumps(DEMONSTRATION_POLICY, sort_keys=True, separators=(",", ":"))
FROZEN_POLICY_HASH = sha256(FROZEN_POLICY_JSON.encode()).hexdigest()

support_table = pd.DataFrame([
    {"plan": name, "support_aware_score": support_aware_score(plan, DEMONSTRATION_POLICY["support_penalty"])[0],
     "unsupported_steps": support_aware_score(plan, DEMONSTRATION_POLICY["support_penalty"])[1]}
    for name, plan in candidate_plans.items()
]).sort_values("support_aware_score", ascending=False)
supported_choice = support_table.iloc[0]["plan"]
supported_truth = rollout_grid(candidate_plans[supported_choice], true_plan_step)
support_aware_mitigation = {
    "selected_plan": supported_choice,
    "realized_goal_reached": (supported_truth[-1].x, supported_truth[-1].y) == GOAL,
    "realized_collision": supported_truth[-1].collided,
    "policy_hash": FROZEN_POLICY_HASH,
    "claim": "mitigates this controlled exploit; does not certify safety",
}
assert supported_choice == "supported_detour"
assert support_aware_mitigation["realized_goal_reached"] and not support_aware_mitigation["realized_collision"]
support_table, support_aware_mitigation


## 18. Site C dynamics shift after policy freeze

Site C changes friction and actuator gain. No threshold, model, support map, or candidate set changes after opening it.


In [ ]:
policy_hash_before_site_c = FROZEN_POLICY_HASH
one_step_site_c = evaluate_one_step(action_model, site_c)
rollout_metrics_c = evaluate_rollout(action_model, SITE_C_DYNAMICS, seed=91)
policy_hash_after_site_c = sha256(json.dumps(DEMONSTRATION_POLICY, sort_keys=True, separators=(",", ":")).encode()).hexdigest()
assert policy_hash_before_site_c == policy_hash_after_site_c
site_c_results = {
    "role": "reporting_only_no_changes",
    "dynamics_shift": "higher inertia persistence and actuator gain",
    "one_step": one_step_site_c,
    "rollout_h20": rollout_metrics_c.iloc[-1].to_dict(),
    "policy_hash_unchanged": True,
}
pd.concat([
    rollout_metrics_b.assign(site="Site B"),
    rollout_metrics_c.assign(site="Site C"),
])


Site C degradation is evidence of a dynamics-shift boundary, not an invitation to tune the release policy on the test source.

## 19. Earliest-failure attribution


In [ ]:
failure_attribution = pd.DataFrame([
    {"case_id": "occlusion_memory", "earliest_failure": "none", "downstream_outcome": "identity retained"},
    {"case_id": "action_ignorant_baseline", "earliest_failure": "action_conditioning_failure", "downstream_outcome": "counterfactuals collapse"},
    {"case_id": "nominal_shortcut", "earliest_failure": "uncertainty_failure", "downstream_outcome": "planning_model_exploitation"},
    {"case_id": "site_c_shift", "earliest_failure": "distribution_shift", "downstream_outcome": "rollout_drift"},
])
failure_taxonomy = [
    "state_estimation_failure", "action_conditioning_failure", "one_step_dynamics_failure",
    "rollout_drift", "object_identity_failure", "physical_constraint_failure",
    "uncertainty_failure", "planning_model_exploitation", "distribution_shift",
]
failure_attribution


## 20. Optional 4D, Dreamer, Cosmos, and Genie paths

All optional paths are disabled. Source pins do not pin model weights, containers, datasets, or transitive licenses. Genie 3 is an official case study rather than an executable repository dependency.


In [ ]:
CV_ENABLE_DREAMERV3=False
CV_ENABLE_COSMOS3=False
CV_ENABLE_DYNAMIC_3D_GAUSSIANS=False

OPTIONAL_INTEGRATIONS = {
    "dreamerv3": {
        "enabled": CV_ENABLE_DREAMERV3,
        "revision": "e3f02248693a79dc8b0ebd62c93683888ddaccfe",
        "license": "MIT",
        "role": "latent dynamics plus imagination case study",
    },
    "cosmos3": {
        "enabled": CV_ENABLE_COSMOS3,
        "revision": "5a68d9d4d34c9ca2bdcb0a1d9bbbb3a2d1b8d497",
        "license": "OpenMDW-1.1; review model cards and downstream assets",
        "role": "optional physical-AI reasoning/world/action system",
    },
    "dynamic_3d_gaussians": {
        "enabled": CV_ENABLE_DYNAMIC_3D_GAUSSIANS,
        "revision": "7dbbd4dec404308524ff402756bdb8143a2589b0",
        "license": "mixed: repository MIT portions plus restrictive Inria dependencies",
        "role": "optional persistent 4D scene mapping",
    },
    "genie3": {
        "enabled": False,
        "revision": "official-product-page-reviewed-2026-09-13",
        "license": "service/research access; no course dependency",
        "role": "closed interactive-world case study",
    },
}
assert not any(item["enabled"] for item in OPTIONAL_INTEGRATIONS.values())
OPTIONAL_INTEGRATIONS


## 21. Governed evidence artifact

The artifact records schemas, environment/model revisions, denominators, rollout mode, counterfactuals, support policy, predicted-versus-realized planning evidence, optional-system governance, and unresolved production assumptions. `authorization` remains `none`.


In [ ]:
evidence = {
    "course": "Advanced 03 — Dynamic Scenes & World Models",
    "authorization": "none",
    "environment_contract": {
        "sites": {"A": SITE_A_DYNAMICS.environment_version, "B": SITE_B_DYNAMICS.environment_version, "C": SITE_C_DYNAMICS.environment_version},
        "bounds_m": WORKCELL_BOUNDS,
        "policy_frozen_before_site_c": True,
        "policy_hash": FROZEN_POLICY_HASH,
    },
    "state_contract": {"schema": "WorldState/ObjectState", "position_unit": "metre", "velocity_unit": "metre_per_second", "identity_scope": "known_sensor_facing_ids_not_reidentification"},
    "state_estimation_contract": {"velocity_dt": "current_observation_timestamp_minus_previous_object_observation_timestamp", "non_increasing_timestamp": "raise_ValueError", "elapsed_time_test": elapsed_time_velocity_test},
    "action_contract": {"schema": "Action", "allowed_kinds": sorted(ACTION_KINDS), "trained_kinds": sorted(MODEL_ACTIONS), "physical_execution": False},
    "observation_contract": {"schema": "Observation", "hidden_velocity_excluded": True, "future_observations_excluded": True},
    "world_model_contract": {"engine": action_model.engine, "foundation_model": action_model.foundation_model, "rollout_mode": "open_loop_recursive"},
    "one_step_metrics": one_step_comparison.to_dict(orient="records"),
    "valve_transition_metrics": valve_transition_report.to_dict(orient="records"),
    "rollout_metrics": {
        "site_b": rollout_metrics_b.to_dict(orient="records"),
        "site_c": rollout_metrics_c.to_dict(orient="records"),
    },
    "action_conditioning": action_conditioning,
    "counterfactual_tests": counterfactual_matrix.to_dict(orient="records"),
    "object_permanence": {"claim": "known_identity_persistence", "reidentification_evaluated": False, "rows": known_identity_persistence_report.to_dict(orient="records")},
    "physics_checks": physics_summary,
    "event_timing": time_to_event_metrics,
    "uncertainty_metrics": {"support_signal": "discrete observed-cell membership", "calibrated_probability": False},
    "stochastic_futures": {"known_answer_metrics": known_answer_metrics, "noisy_model_metrics": stochastic_future_metrics, "frequency_table": stochastic_frequency_table.to_dict(orient="records")},
    "planning_results": {"nominal": model_exploitation_test, "support_aware": support_aware_mitigation},
    "model_exploitation_test": model_exploitation_test,
    "site_c_results": site_c_results,
    "optional_model_observations": OPTIONAL_INTEGRATIONS,
    "versions": VERSIONS,
    "unresolved_production_assumptions": [
        "toy dynamics are not contact-rich physics",
        "support membership is not calibrated epistemic uncertainty",
        "no hardware-in-the-loop or sim-to-real validation",
        "no physical action is authorized",
    ],
}
assert evidence["authorization"] == "none"
artifact_dir = Path(".artifacts") / "advanced-03-dynamic-scenes-world-models"
artifact_dir.mkdir(parents=True, exist_ok=True)
(artifact_dir / "world_model_evidence.json").write_text(json.dumps(evidence, indent=2, default=float), encoding="utf-8")
decision = pd.DataFrame([{
    "decision": "research_only_no_physical_authorization",
    "site_b_position_vector_RMSE_m": one_step_comparison.loc[0, "position_vector_RMSE_m"],
    "site_c_position_vector_RMSE_m": one_step_site_c["position_vector_RMSE_m"],
    "nominal_plan_collision": model_exploitation_test["realized_collision"],
    "support_mitigation_success": support_aware_mitigation["realized_goal_reached"],
    "policy_hash": FROZEN_POLICY_HASH,
}])
decision.to_csv(artifact_dir / "world_model_decision.csv", index=False)
decision


## 22. Production upgrade and exercises

| Teaching element | Production upgrade |
| --- | --- |
| explicit toy state | versioned state/action schemas, frame graphs, uncertainty |
| deterministic simulator | calibrated digital twin and hardware-in-the-loop evidence |
| Ridge proxy | sequence/latent model with held-out interventions and embodiments |
| support cells | calibrated ensembles, density or conformal support plus abstention |
| enumerated planner | bounded MPC with hard constraints, budgets, authorization, and rollback |
| local artifacts | immutable lineage, approvals, drift alerts, replay, and incident response |

Exercises:

1. Add a hidden actuator mode and delayed valve transition, then prove that one frame is non-Markov.
2. Replace Ridge with a small neural sequence model while preserving evaluation isolation.
3. Compare interpolated event timing with discrete-step timing and add interval-censoring policy.
4. Add a second occluded object and quantify uncertain identity.
5. Replan after each true step without giving the planner hidden dynamics.
6. Design a dynamic-Gaussian adapter that preserves object and camera frames.


## 23. Explain without code

You should now be able to explain why observation differs from state; why elapsed observation time matters for velocity; why known-ID persistence is not re-identification; why action conditioning is more than text prompting; why continuous and discrete state need separate metrics; why one-step accuracy hides rollout drift; why multiple futures can be valid yet miscalibrated; why model response is not validated causality; why a planner exploits model error; why support-aware scoring mitigates but does not certify; and why a generated video can never authorize an actuator.


In [ ]:
assert evidence["world_model_contract"]["engine"] == "local_world_model_proxy"
assert evidence["observation_contract"]["hidden_velocity_excluded"]
assert FROZEN_POLICY_HASH == policy_hash_after_site_c
assert model_exploitation_test["realized_collision"]
assert support_aware_mitigation["realized_goal_reached"]
assert stochastic_future_metrics["valid_mode_coverage"] == 1.0
assert known_identity_persistence_report["cart_retained"].all()
assert valve_transition_report.loc[0, "valve_state_accuracy"] >= 0.95
print("Advanced 03 invariant checks passed; artifacts:", artifact_dir)
